# Tire Analysis: Temperature Distribution & Thermography

This notebook visualizes tire temperature data from infrared sensors across your tires, helping you understand tire behavior and identify setup issues.

## What You'll Find Here

- **Tire Temperature Heatmaps**: Visual representation of temperature distribution across all four tires (FL, FR, RL, RR) throughout the lap
- **Temperature vs Distance**: See how tire temperatures change through different track sections
- **Speed & G-Force Overlay**: Correlate tire temperatures with speed and combined lateral/longitudinal acceleration
- **Driver Inputs Overlay**: See throttle, brake, and steering inputs aligned with temperature data

## How to Interpret the Results

### Temperature Heatmaps
- **Ch1-Ch8**: Represent temperature sensor positions across the tire width
  - FL/RL: Ch1 = Outside (left), Ch8 = Inside (right)
  - FR/RR: Ch1 = Inside (left), Ch8 = Outside (right)
- **Hot spots (bright)**: Areas of high temperature - could indicate excessive load or slip
- **Cold spots (dark)**: Areas not being worked - potential grip left on the table
- **Even temperature gradient**: Indicates good tire usage and camber settings

### Setup Insights
- **Outside edge hot**: May need more negative camber
- **Inside edge hot**: May have too much negative camber
- **Center hot**: Could indicate over-inflation
- **Edges hot, center cold**: Could indicate under-inflation
- **Front vs Rear difference**: Balance insights for understeer/oversteer tendencies

## Using Your Own Data

To analyze your own data:

1. **Run the first cell** below to install packages and display the upload widget
2. **Click "Choose File"** to select your `.xrk` or `.xrz` file
3. **Run all remaining cells** to analyze your data

The status indicator will show which file is being used. If you don't upload a file, the sample data will be used.

## Requirements

- Tire temperature channels (`FL_Ch1`-`FL_Ch8`, `FR_Ch1`-`FR_Ch8`, `RL_Ch1`-`RL_Ch8`, `RR_Ch1`-`RR_Ch8`)
- GPS Speed channel for distance calculation
- Acceleration data (`LateralAcc`, `InlineAcc`) for G-force visualization
- Driver input channels (`BrakePress`, `PPS`, `SteerAngle`)

**Note:** This notebook works in both JupyterLite (browser) and standard JupyterLab environments.

In [1]:
# Install required packages (needed for JupyterLite, skipped in regular JupyterLab if already installed)
%pip install -q pandas plotly libxrk motorsports-data-notebook jinja2 ipywidgets

# Import helper functions
from motorsports_data_notebook.channels import get_best_lap_channels
from motorsports_data_notebook.visualization import (
    format_lap_time,
    plot_tire_thermography,
    show_fig,
)
from motorsports_data_notebook.widgets import FileUpload, load_session

# File picker - upload your own .xrk/.xrz file or use the sample data
file_upload = FileUpload(default_file="CMD_Inferno 86_Fuji GP Sh_Generic testing_a_2248.xrz")
file_upload.display()

In [ ]:
# Load the data file with derived columns (speed_kmh, distance_m, lap_time)
log = load_session(file_upload.get_file_data())

# Get laps as pandas DataFrame for display
laps = log.laps.to_pandas()

In [5]:
# Display lap times table
laps.style.format({"lap_time": format_lap_time})

In [6]:
# Define channels needed for tire thermography (only interpolate what we need)
tire_channels = [f"{pos}_Ch{i}" for pos in ["FL", "FR", "RL", "RR"] for i in range(1, 9)]
other_channels = [
    "distance_m",
    "speed_kmh",
    "LateralAcc",
    "InlineAcc",
    "BrakePress",
    "PPS",
    "SteerAngle",
]

# Extract best lap data at native sample rates
best_lap, channels = get_best_lap_channels(log, laps, tire_channels + other_channels)

In [7]:
# Tire Thermography - Best Lap (uses on-demand interpolation)
fig = plot_tire_thermography(channels, title="Tire Temperatures - Best Lap")
show_fig(fig)